# Incident Root Cause Analyzer — Training Analysis

This notebook visualises the fine-tuning progress, evaluates the trained model, and explores the dataset distribution.

**Sections:**
1. Dataset Exploration
2. Training Loss Curves
3. Evaluation Metrics
4. Per-Category Performance
5. Example Predictions

In [ ]:
import json
import os
import sys
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

plt.style.use('dark_background')
ACCENT = '#5865f2'
GREEN  = '#23d160'
RED    = '#ff3860'
YELLOW = '#ffdd57'
COLORS = [ACCENT, GREEN, '#ff9f43', RED, '#a29bfe', '#fd79a8', '#00cec9', '#e17055', '#74b9ff', '#55efc4']

## 1. Dataset Exploration

In [ ]:
DATA_PATH = '../data/training_incidents.jsonl'
TEST_PATH = '../data/training_incidents_test.jsonl'

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_data = load_jsonl(DATA_PATH)
test_data  = load_jsonl(TEST_PATH)
df_train   = pd.DataFrame(train_data)
df_test    = pd.DataFrame(test_data)

print(f'Train examples: {len(df_train)}')
print(f'Test  examples: {len(df_test)}')
print(f'Columns: {list(df_train.columns)}')
df_train[['service','severity','category']].head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Dataset Distribution', fontsize=14, fontweight='bold')

# Category distribution
cat_counts = df_train['category'].value_counts()
axes[0].barh(cat_counts.index, cat_counts.values, color=COLORS[:len(cat_counts)])
axes[0].set_title('Category Distribution (Train)')
axes[0].set_xlabel('Count')

# Severity distribution
sev_counts = df_train['severity'].value_counts()
axes[1].pie(sev_counts.values, labels=sev_counts.index, colors=[RED, YELLOW, ACCENT],
            autopct='%1.0f%%', startangle=90)
axes[1].set_title('Severity Distribution')

# Service distribution (top 10)
svc_counts = df_train['service'].value_counts().head(10)
axes[2].barh(svc_counts.index, svc_counts.values, color=GREEN)
axes[2].set_title('Top 10 Services')
axes[2].set_xlabel('Count')

plt.tight_layout()
plt.savefig('dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Token length distribution
df_train['input_len'] = df_train['input'].apply(lambda x: len(str(x).split()))
df_train['output_len'] = df_train['output'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_train['input_len'], bins=40, color=ACCENT, alpha=0.8, edgecolor='none')
axes[0].set_title('Input Length (words)')
axes[0].set_xlabel('Word count')
axes[0].axvline(df_train['input_len'].median(), color=GREEN, linestyle='--', label=f"Median: {df_train['input_len'].median():.0f}")
axes[0].legend()

axes[1].hist(df_train['output_len'], bins=30, color=GREEN, alpha=0.8, edgecolor='none')
axes[1].set_title('Output Length (words)')
axes[1].set_xlabel('Word count')
axes[1].axvline(df_train['output_len'].median(), color=ACCENT, linestyle='--', label=f"Median: {df_train['output_len'].median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.savefig('token_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Input  — mean: {df_train['input_len'].mean():.0f}, p95: {df_train['input_len'].quantile(0.95):.0f}")
print(f"Output — mean: {df_train['output_len'].mean():.0f}, p95: {df_train['output_len'].quantile(0.95):.0f}")

## 2. Training Loss Curves

In [ ]:
METRICS_PATH = '../train/checkpoints/incident-analyzer-v1/training_metrics.json'

if os.path.exists(METRICS_PATH):
    with open(METRICS_PATH) as f:
        metrics = json.load(f)
    
    train_steps = [m['step'] for m in metrics['training_losses']]
    train_losses = [m['loss'] for m in metrics['training_losses']]
    
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(train_steps, train_losses, color=ACCENT, linewidth=1.5, label='Training Loss')
    
    if metrics.get('eval_losses'):
        eval_steps = [m['step'] for m in metrics['eval_losses']]
        eval_losses = [m['eval_loss'] for m in metrics['eval_losses']]
        ax.plot(eval_steps, eval_losses, color=GREEN, linewidth=2, marker='o', label='Eval Loss')
    
    # Smooth training loss
    window = max(1, len(train_losses) // 20)
    smoothed = pd.Series(train_losses).rolling(window, center=True).mean()
    ax.plot(train_steps, smoothed, color=YELLOW, linewidth=2.5, linestyle='--', label=f'Smoothed (w={window})')
    
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Cross-Entropy Loss')
    ax.set_title('QLoRA Fine-tuning Loss Curve — Mistral-7B on Incident Data')
    ax.legend()
    ax.grid(True, alpha=0.1)
    
    final_loss = train_losses[-1]
    ax.annotate(f'Final: {final_loss:.3f}', 
                xy=(train_steps[-1], final_loss),
                xytext=(-60, 15), textcoords='offset points',
                color=YELLOW, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=YELLOW))
    
    plt.tight_layout()
    plt.savefig('training_loss_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'Initial loss: {train_losses[0]:.4f}')
    print(f'Final loss:   {train_losses[-1]:.4f}')
    print(f'Improvement:  {(1 - train_losses[-1]/train_losses[0])*100:.1f}%')
else:
    print(f'No training metrics found at {METRICS_PATH}')
    print('Run train.py first, then re-run this cell.')
    
    # Generate mock curve for illustration
    np.random.seed(42)
    steps = np.arange(0, 500, 5)
    base_loss = 2.8 * np.exp(-0.007 * steps) + 0.35 + np.random.normal(0, 0.05, len(steps))
    
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(steps, base_loss, color=ACCENT, alpha=0.5, linewidth=1)
    ax.plot(steps, pd.Series(base_loss).rolling(20, center=True).mean(), color=YELLOW, linewidth=2.5, label='Smoothed loss (illustrative)')
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Loss')
    ax.set_title('Expected Training Loss Curve (Illustrative — run training to see real curve)')
    ax.legend()
    ax.grid(True, alpha=0.1)
    ax.text(0.5, 0.5, 'ILLUSTRATIVE ONLY', transform=ax.transAxes,
            ha='center', va='center', fontsize=24, alpha=0.3, color=RED, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 3. Evaluation Metrics

In [ ]:
EVAL_RESULTS_PATH = '../train/eval_results.json'

if os.path.exists(EVAL_RESULTS_PATH):
    with open(EVAL_RESULTS_PATH) as f:
        eval_results = json.load(f)
    
    print('=== EVALUATION RESULTS ===')
    print(f"Category Accuracy:  {eval_results['category_accuracy']:.1%}")
    print(f"F1 Macro:           {eval_results['f1_macro']:.4f}")
    print(f"F1 Weighted:        {eval_results['f1_weighted']:.4f}")
    print(f"ROUGE-1:            {eval_results['rouge']['rouge1']:.4f}")
    print(f"ROUGE-2:            {eval_results['rouge']['rouge2']:.4f}")
    print(f"ROUGE-L:            {eval_results['rouge']['rougeL']:.4f}")
    print(f"Test examples:      {eval_results['num_examples']}")
    
    # Radar chart
    metrics_names = ['Accuracy', 'F1 Macro', 'F1 Weighted', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L']
    metrics_vals  = [
        eval_results['category_accuracy'],
        eval_results['f1_macro'],
        eval_results['f1_weighted'],
        eval_results['rouge']['rouge1'],
        eval_results['rouge']['rouge2'],
        eval_results['rouge']['rougeL'],
    ]
    target_vals = [0.80, 0.75, 0.80, 0.70, 0.40, 0.65]

    N = len(metrics_names)
    angles = [n / float(N) * 2 * np.pi for n in range(N)] + [0]
    vals_plot = metrics_vals + [metrics_vals[0]]
    target_plot = target_vals + [target_vals[0]]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    ax.fill(angles, vals_plot, alpha=0.25, color=ACCENT)
    ax.plot(angles, vals_plot, color=ACCENT, linewidth=2, label='Model')
    ax.plot(angles, target_plot, color=GREEN, linewidth=1.5, linestyle='--', label='Target')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics_names, size=11)
    ax.set_ylim(0, 1)
    ax.set_title('Model Performance vs Targets', pad=20, fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
    plt.tight_layout()
    plt.savefig('evaluation_radar.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No evaluation results found.')
    print('Run: python train/eval.py --model_dir train/checkpoints/incident-analyzer-v1/final')

## 4. Per-Category Performance

In [ ]:
if os.path.exists(EVAL_RESULTS_PATH):
    per_cat = eval_results.get('per_category_accuracy', {})
    cats = list(per_cat.keys())
    accs = list(per_cat.values())
    
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = [GREEN if a >= 0.80 else YELLOW if a >= 0.60 else RED for a in accs]
    bars = ax.barh(cats, accs, color=colors, alpha=0.85)
    ax.axvline(0.80, color=GREEN, linestyle='--', linewidth=1.5, label='80% target')
    
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{acc:.0%}', va='center', fontsize=10)
    
    ax.set_xlim(0, 1.1)
    ax.set_xlabel('Accuracy')
    ax.set_title('Per-Category Classification Accuracy')
    ax.legend()
    ax.grid(True, axis='x', alpha=0.15)
    plt.tight_layout()
    plt.savefig('per_category_accuracy.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Example Predictions (Live Inference)

In [ ]:
import httpx

API_URL = 'http://localhost:8000'

SAMPLE_INCIDENTS = [
    {
        'name': 'N+1 Query',
        'logs': '[2024-01-15 14:23:45] ERROR checkout-service: 248 queries in single request\n[2024-01-15 14:23:45] DEBUG ORM: SELECT * FROM orders WHERE id=1; (2ms)\n[2024-01-15 14:23:45] ERROR checkout-service: timeout 45000ms',
        'metrics': 'DB CPU: 98%, Connection pool: 100%, P99 latency: 45000ms',
        'service': 'checkout-service',
    },
    {
        'name': 'Memory Leak',
        'logs': '[2024-01-15 10:15:00] WARN payment-service: Heap 7400MB\n[2024-01-15 10:15:02] ERROR k8s: OOMKilled (exit 137)\n[2024-01-15 10:15:03] INFO k8s: Restart #8',
        'metrics': 'Heap: 7400/8192MB, GC: 35% CPU, Restarts: 8/hr',
        'service': 'payment-service',
    },
    {
        'name': 'Disk Full',
        'logs': '[2024-01-16 08:30:00] ERROR api-gateway: Write failed: No space left on device (/var/log)\n[2024-01-16 08:30:01] ERROR postgres: could not write to pg_wal',
        'metrics': 'Disk /var/log: 99% used, Available: 120MB',
        'service': 'api-gateway',
    },
]

for inc in SAMPLE_INCIDENTS:
    print(f"\n{'='*60}")
    print(f"Incident: {inc['name']}")
    print(f"Service:  {inc['service']}")
    
    try:
        res = httpx.post(
            f'{API_URL}/analyze',
            json={'logs': inc['logs'], 'metrics': inc['metrics'], 'service': inc['service']},
            timeout=120,
        )
        if res.status_code == 200:
            data = res.json()
            print(f"Root Cause:  {data['root_cause']}")
            print(f"Confidence:  {data['confidence']:.0%}")
            print(f"Category:    {data['category']}")
            print(f"Infer time:  {data['inference_time_ms']}ms")
            print("Fix Steps:")
            for i, step in enumerate(data['fix_steps'][:3], 1):
                print(f"  {i}. {step}")
        else:
            print(f"Error: HTTP {res.status_code} — {res.text[:200]}")
    except Exception as e:
        print(f"Could not reach API at {API_URL}: {e}")
        print('Start the API first: cd api && uvicorn main:app --reload')